## meta 데이터 시계열 파생변수 추가

In [ ]:
# ==============================================
#  FINAL VERSION
# ==============================================

import numpy as np
import pandas as pd
from xgboost import XGBRegressor
from google.colab import files
import re


uploaded = files.upload()


👉 모든 파일 업로드


Saving TEST_전국도매_23.csv to TEST_전국도매_23.csv
Saving TEST_전국도매_24.csv to TEST_전국도매_24.csv
Saving train.csv to train.csv
Saving TRAIN_산지공판장_2018-2021.csv to TRAIN_산지공판장_2018-2021.csv
Saving TRAIN_전국도매_2018-2021.csv to TRAIN_전국도매_2018-2021.csv
Saving TEST_산지공판장_21.csv to TEST_산지공판장_21.csv
Saving TEST_산지공판장_22.csv to TEST_산지공판장_22.csv
Saving TEST_산지공판장_23.csv to TEST_산지공판장_23.csv
Saving TEST_산지공판장_24.csv to TEST_산지공판장_24.csv
Saving TEST_전국도매_00.csv to TEST_전국도매_00.csv
Saving TEST_전국도매_01.csv to TEST_전국도매_01.csv
Saving TEST_전국도매_02.csv to TEST_전국도매_02.csv
Saving TEST_전국도매_03.csv to TEST_전국도매_03.csv
Saving TEST_전국도매_04.csv to TEST_전국도매_04.csv
Saving TEST_전국도매_05.csv to TEST_전국도매_05.csv
Saving TEST_전국도매_06.csv to TEST_전국도매_06.csv
Saving TEST_전국도매_07.csv to TEST_전국도매_07.csv
Saving TEST_전국도매_08.csv to TEST_전국도매_08.csv
Saving TEST_전국도매_09.csv to TEST_전국도매_09.csv
Saving TEST_전국도매_10.csv to TEST_전국도매_10.csv
Saving TEST_전국도매_11.csv to TEST_전국도매_11.csv
Saving TEST_전국도매_12.csv to TEST_전국도매_12.csv
Savi

In [ ]:
import re

# ======================
# 파일 리스트 먼저 만들기
# ======================
files_list = list(uploaded.keys())

print("📂 업로드된 파일:")
for f in files_list:
    print(f)

# ======================
# train 파일
# ======================
train_file = [f for f in files_list if 'train' in f.lower() and 'test' not in f.lower()][0]

# ======================
# train 메타데이터
# ======================
gong_train_file = [f for f in files_list if '산지공판장' in f and 'TRAIN' in f][0]
whole_train_file = [f for f in files_list if '전국도매' in f and 'TRAIN' in f][0]

# ======================
# sample submission
# ======================
sample_file = [f for f in files_list if 'sample' in f.lower()][0]

# ======================
# test 파일 분리
# ======================
price_tests = {}
gong_tests = {}
whole_tests = {}

for f in files_list:
    if 'TEST_' in f:

        nums = re.findall(r'\d+', f)
        if len(nums) == 0:
            continue

        idx = nums[0]  # 00, 01, ... 24

        if '산지공판장' in f:
            gong_tests[idx] = f
        elif '전국도매' in f:
            whole_tests[idx] = f
        else:
            price_tests[idx] = f

# ======================
# 매칭 확인 (디버깅용)
# ======================
print("\n✅ 매칭 확인")
for k in sorted(price_tests.keys()):
    print(f"{k} → {price_tests[k]} | {gong_tests.get(k)} | {whole_tests.get(k)}")

📂 업로드된 파일:
TEST_전국도매_23.csv
TEST_전국도매_24.csv
train.csv
TRAIN_산지공판장_2018-2021.csv
TRAIN_전국도매_2018-2021.csv
TEST_산지공판장_21.csv
TEST_산지공판장_22.csv
TEST_산지공판장_23.csv
TEST_산지공판장_24.csv
TEST_전국도매_00.csv
TEST_전국도매_01.csv
TEST_전국도매_02.csv
TEST_전국도매_03.csv
TEST_전국도매_04.csv
TEST_전국도매_05.csv
TEST_전국도매_06.csv
TEST_전국도매_07.csv
TEST_전국도매_08.csv
TEST_전국도매_09.csv
TEST_전국도매_10.csv
TEST_전국도매_11.csv
TEST_전국도매_12.csv
TEST_전국도매_13.csv
TEST_전국도매_14.csv
TEST_전국도매_15.csv
TEST_전국도매_16.csv
TEST_전국도매_17.csv
TEST_전국도매_18.csv
TEST_전국도매_19.csv
TEST_전국도매_20.csv
TEST_전국도매_21.csv
TEST_전국도매_22.csv
sample_submission.csv
TEST_00.csv
TEST_01.csv
TEST_02.csv
TEST_03.csv
TEST_04.csv
TEST_05.csv
TEST_06.csv
TEST_07.csv
TEST_08.csv
TEST_09.csv
TEST_10.csv
TEST_11.csv
TEST_12.csv
TEST_13.csv
TEST_14.csv
TEST_15.csv
TEST_16.csv
TEST_17.csv
TEST_18.csv
TEST_19.csv
TEST_20.csv
TEST_21.csv
TEST_22.csv
TEST_23.csv
TEST_24.csv
TEST_산지공판장_00.csv
TEST_산지공판장_01.csv
TEST_산지공판장_02.csv
TEST_산지공판장_03.csv
TEST_산지공판장_04.csv
TEST_산지공판장_05.csv
T

In [ ]:
train = pd.read_csv(train_file)
gong_train = pd.read_csv(gong_train_file)
whole_train = pd.read_csv(whole_train_file)

In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor

# ======================
# 전처리 함수
# ======================
def clean(df):
    df.columns = df.columns.str.strip()
    return df

# ======================
# 시간 파싱 (train)
# ======================
def parse_time(x):
    x=str(x)
    year=int(x[:4]); month=int(x[4:6])
    if '상순' in x: p=1
    elif '중순' in x: p=2
    else: p=3
    return pd.Series([year,month,p])

# ======================
# TEST 시간 파싱
# ======================
def parse_t(x):
    x = str(x).replace('순','')
    if x == 'T': return 0
    elif 'T-' in x: return int(x.replace('T',''))
    else: return 0

# ======================
# TARGET MAP
# ======================
TARGET_MAP = {
"건고추":{"품종명":["화건"]},
"사과":{"품종명":["홍로","후지"]},
"감자":{"품종명":["감자 수미"]},
"배":{"품종명":["신고"]},
"깐마늘(국산)":{"품종명":["깐마늘(국산)"]},
"무":{"품종명":["무"]},
"상추":{"품종명":["청"]},
"배추":{"품종명":["배추"]},
"양파":{"품종명":["양파"]},
"대파":{"품종명":["대파(일반)"]},
}

# ======================
# feature 함수
# ======================
def make_features(df):
    df=df.sort_values(['year','month','period'])

    for i in range(1,9):
        df[f'lag_{i}']=df['평균가격(원)'].shift(i)

    df['diff_1']=df['평균가격(원)'].diff(1)
    df['diff_3']=df['평균가격(원)'].diff(3)

    df['rolling_mean_3']=df['평균가격(원)'].shift(1).rolling(3).mean()
    df['rolling_mean_5']=df['평균가격(원)'].shift(1).rolling(5).mean()
    df['rolling_std_5']=df['평균가격(원)'].shift(1).rolling(5).std()

    df['gong_lag_1']=df['gong_mean_price'].shift(1)
    df['whole_lag_1']=df['whole_mean_price'].shift(1)

    df['gong_vol_diff']=df['gong_volume'].diff(1)
    df['whole_vol_diff']=df['whole_volume'].diff(1)

    return df

# ======================
# FEATURES
# ======================
FEATURES = [
    '평균가격(원)', '평년 평균가격(원)',
    'year','month','period',
    'gong_mean_price','gong_volume',
    'whole_mean_price','whole_volume',
    'spread','price_per_volume',
    'diff_1','diff_3',
    'rolling_mean_3','rolling_mean_5','rolling_std_5',
    'gong_lag_1','whole_lag_1',
    'gong_vol_diff','whole_vol_diff'
]

for i in range(1,9):
    FEATURES.append(f'lag_{i}')

# ======================
# 데이터 로드 (이미 있음 가정)
# ======================
train = clean(train)
gong_train = clean(gong_train)
whole_train = clean(whole_train)

for df in [train,gong_train,whole_train]:
    df[['year','month','period']] = df['시점'].apply(parse_time)

gong_train['price'] = gong_train['평균가(원/kg)']
whole_train['price'] = whole_train['평균가(원/kg)']

gong_feat = gong_train.groupby(['품목명','year','month','period']).agg({
    'price':'mean','총반입량(kg)':'sum'
}).reset_index()
gong_feat.columns=['품목명','year','month','period','gong_mean_price','gong_volume']

whole_feat = whole_train.groupby(['품목명','year','month','period']).agg({
    'price':'mean','총반입량(kg)':'sum'
}).reset_index()
whole_feat.columns=['품목명','year','month','period','whole_mean_price','whole_volume']

# ======================
# TRAIN
# ======================
models={}

for item in TARGET_MAP:
    df=train[train['품목명']==item]

    df=df.merge(gong_feat,on=['품목명','year','month','period'],how='left')
    df=df.merge(whole_feat,on=['품목명','year','month','period'],how='left')

    df['spread']=df['whole_mean_price']-df['gong_mean_price']
    df['price_per_volume']=df['평균가격(원)']/(df['gong_volume']+1)

    df=df.groupby('품종명').apply(make_features).reset_index(drop=True)

    df['target_h1']=df['평균가격(원)'].shift(-1)
    df['target_h2']=df['평균가격(원)'].shift(-2)
    df['target_h3']=df['평균가격(원)'].shift(-3)

    item_models={}
    for h in [1,2,3]:
        temp=df.dropna(subset=[f'target_h{h}'])
        X=temp[FEATURES].fillna(0)
        y=temp[f'target_h{h}']

        model=XGBRegressor(n_estimators=800,max_depth=6,learning_rate=0.03)
        model.fit(X,y)
        item_models[h]=model

    models[item]=item_models

# ======================
# TEST
# ======================
rows=[]

for idx in sorted(price_tests.keys()):
    price=pd.read_csv(price_tests[idx])
    gong=pd.read_csv(gong_tests[idx])
    whole=pd.read_csv(whole_tests[idx])

    price=clean(price); gong=clean(gong); whole=clean(whole)

    for df in [price,gong,whole]:
        df['t_idx']=df['시점'].apply(parse_t)

    base=pd.Timestamp(2000,1,1)
    for df in [price,gong,whole]:
        df['date']=df['t_idx'].apply(lambda x: base+pd.Timedelta(days=10*x))
        df['year']=df['date'].dt.year
        df['month']=df['date'].dt.month
        df['period']=(df['date'].dt.day//10)+1

    gong['price']=gong['평균가(원/kg)']
    whole['price']=whole['평균가(원/kg)']

    gong_feat_t=gong.groupby(['품목명','year','month','period']).agg({
        'price':'mean','총반입량(kg)':'sum'
    }).reset_index()
    gong_feat_t.columns=['품목명','year','month','period','gong_mean_price','gong_volume']

    whole_feat_t=whole.groupby(['품목명','year','month','period']).agg({
        'price':'mean','총반입량(kg)':'sum'
    }).reset_index()
    whole_feat_t.columns=['품목명','year','month','period','whole_mean_price','whole_volume']

    for item in TARGET_MAP:
        temp=price[price['품목명']==item]
        if len(temp)==0: continue

        temp=temp.merge(gong_feat_t,on=['품목명','year','month','period'],how='left')
        temp=temp.merge(whole_feat_t,on=['품목명','year','month','period'],how='left')

        temp['spread']=temp['whole_mean_price']-temp['gong_mean_price']
        temp['price_per_volume']=temp['평균가격(원)']/(temp['gong_volume']+1)

        temp=make_features(temp)
        last=temp.iloc[-1:]

        for h in [1,2,3]:
            pred=models[item][h].predict(last[FEATURES].fillna(0))

            # 🔥 음수 제거
            pred=np.maximum(pred,0)

            rows.append({
                '시점':f'TEST_{idx}+{h}',
                '품목명':item,
                '값':float(pred)
            })



/tmp/ipykernel_7686/2413154036.py:127: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df=df.groupby('품종명').apply(make_features).reset_index(drop=True)
/tmp/ipykernel_7686/2413154036.py:127: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df=df.groupby('품종명').apply(make_features).reset_index(drop=True)
/tmp/ipykernel_7686/2413154036.py:127: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping col

In [ ]:
print(gong_train.columns)
print(whole_train.columns)

Index(['시점', '공판장코드', '공판장명', '품목코드', '품목명', '품종코드', '품종명', '등급코드', '등급명',
       '총반입량(kg)', '총거래금액(원)', '평균가(원/kg)', '중간가(원/kg)', '최저가(원/kg)',
       '최고가(원/kg)', '경매 건수', '전순 평균가격(원) PreVious SOON',
       '전달 평균가격(원) PreVious MMonth', '전년 평균가격(원) PreVious YeaR',
       '평년 평균가격(원) Common Year SOON', '연도', 'year', 'month', 'period'],
      dtype='object')
Index(['시점', '시장코드', '시장명', '품목코드', '품목명', '품종코드', '품종명', '총반입량(kg)',
       '총거래금액(원)', '평균가(원/kg)', '고가(20%) 평균가', '중가(60%) 평균가', '저가(20%) 평균가',
       '중간가(원/kg)', '최저가(원/kg)', '최고가(원/kg)', '경매 건수',
       '전순 평균가격(원) PreVious SOON', '전달 평균가격(원) PreVious MMonth',
       '전년 평균가격(원) PreVious YeaR', '평년 평균가격(원) Common Year SOON', '연도', 'year',
       'month', 'period'],
      dtype='object')


In [ ]:

# ======================
# 제출
# ======================
sample_sub=pd.read_csv(sample_file)

sub=pd.DataFrame(rows)
pivot=sub.pivot(index='시점',columns='품목명',values='값').reset_index()

pivot=pivot.set_index('시점').reindex(sample_sub['시점']).reset_index()
pivot=pivot.reindex(columns=sample_sub.columns)
pivot=pivot.fillna(0)

pivot.to_csv("submission_final.csv",index=False,encoding='utf-8-sig')

print("✅ 제출 파일 생성 완료")

✅ 제출 파일 생성 완료
